# ***Dataset Description***

In [ ]:
# import the libraries neccesary for handling and plotting data
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
pd.set_option("display.max_columns", None)
# read the csv file into a pandas DataFrame, and print the first 5 entries
df = pd.read_csv("diabetes.csv")
df.head()

In [ ]:
print(f"The number of features in the dataset is {len(df.columns)}")
list(df.columns)

In [ ]:
print(f"The number of rows in the dataset is {len(df)}")

In [ ]:
df.describe()

In [ ]:
df.info()

In [ ]:
target = "diabetes_risk_score"
other_target = ["diagnosed_diabetes", "diabetes_stage"]
# The columns for age, gender, ethnicity, education and income level as well employment status are missing some values
# so they should be imputed later on if they will be used
total_entries = len(df)
categories_missing = ["age", "gender", "ethnicity", "education_level", "income_level", "employment_status"]
for category in categories_missing:
  non_null = df[category].count()
  print(f"The category {category} has {total_entries - non_null} missing values out of {total_entries} values")

print()

#furthermore, the age column has values that are unrealistic such as negative values and inf or decimal values, let us check how many are present
print(f"The number of entries with age=inf are {(df["age"] == np.inf).sum()}")
print(f"The number of entries with age negative are {(df["age"] < 0).sum()}")
is_decimal_mask = (df["age"] % 1 != 0) & (~df["age"].isnull()) & (df["age"] != np.inf)
print(f"The number of entries with age as decimal value are {is_decimal_mask.sum()}")

In [ ]:
categorical_features = []

# Loop through all the features
for feature in df.columns:
  # Check the feature type
  feature_type = type(df[feature][0])

  # If the feature type is a string, then it is a categorical feature
  if feature_type is str and feature != "diabetes_stage":
    categorical_features.append(feature)
# the features that denote if the patient's family has a history of diseases are also binary categorical features
categorical_features.extend(["cardiovascular_history","hypertension_history","family_history_diabetes"])
print("The following are categorical features:")
print(categorical_features)
print()

numerical_features = [x for x in df.columns if x not in categorical_features + other_target]
print("the following are numerical features")
print(numerical_features)

# ***Data Visualization***

In [ ]:
import seaborn as sns
new_df = df.copy()
new_df = new_df[(df["age"] != np.inf)]
for feature in numerical_features:
  sns.displot(new_df[feature],kde=True)
  plt.title(f"Histogram of feature {feature}")
  plt.xlabel(feature)
  plt.ylabel("Frequency")
  plt.show()

In [ ]:
df[target].describe()

In [ ]:
# To find class imbalances, lets find the number of each class in a categorical feature
for feature in categorical_features:
  print(f"Number of classes in feature {feature}")
  # unique method finds the unique classes in a column
  classes = df[feature].unique()
  # use the subplot method in matplotlib to plot multiple graphs in one plotting space
  # value_counts finds the count of each unique class in a dataframe/series
  num_unique = df[feature].value_counts()
  # loop over the indices (classes) in the series that has the count of each unique class
  for index in num_unique.index:
    # find the percentage of each class to the total number of entries in the category
    percentage = np.round(num_unique[index]/num_unique.sum() * 100,decimals = 1)
    print(f"{index}: {num_unique[index]} (%{percentage})")
  # plot a bar graph showing the number of each class in a category
  num_unique.plot(kind="bar")
  plt.title(f"Frequency of feature {feature}")
  plt.ylabel("Count")
  plt.show()
  print('\n')

In [ ]:
# use a box plot to see how the mean of target (risk_score) changes with each class in a category
for category in categorical_features + ["alcohol_consumption_per_week"]:
  # use dropna to remove the rows that have missing values in the specific column (category)
  new_df = df.dropna(subset=[category])
  # plot a boxplot
  new_df.boxplot(column="diabetes_risk_score",by=category,ylabel="risk score")
  plt.title(f"boxplot of {category} vs risk score")
  plt.suptitle('')

In [ ]:
features= ['family_history_diabetes']
# let us now turn to the numerical categories, we will now plot scatterplots for all the numerical feature in the dataset
new_df = df.drop(columns=categorical_features + ["alcohol_consumption_per_week","diabetes_stage","diagnosed_diabetes"], axis=1)
# we will drop the rows that have missing or nonsensical age values such as infinity, negative and decimal
new_df = new_df[(new_df["age"] != np.nan) & (new_df["age"] != np.inf) & (new_df["age"] >= 0) & ~is_decimal_mask]

In [ ]:
for feature in new_df.columns:
    new_df.plot.scatter(x=feature,y="diabetes_risk_score")

# ***Correlation analysis and Feature Selection***

In [ ]:
# from the scatter plot, we can see variable that show a trend, such as a negative trend between physical activity and risk score,
# as well as a positive trend in age and glucose_fasting with risk_score
# but, to make sure of the correlation between features and the label, let us find the correlation matrix
corr_mat = new_df.corr()
sns.heatmap(corr_mat)
corr_mat["diabetes_risk_score"].sort_values()

In [ ]:
# find the features that show a correlation with a magnitude of above 0.3 with the target label
numerical_features_interest = corr_mat[(abs(corr_mat[target]) > 0.30)][target]
numerical_features_interestname = numerical_features_interest.index.tolist()
print(numerical_features_interestname)

In [ ]:
# find the correlation between the numerical features we are interested in
new_df[numerical_features_interestname].corr()

In [ ]:
# from the above graph, we can see that some features show collinearilty, which is when dependent features show correlation with each other, rendering one of them as redundant
# the features age and systolic_bp, as well as hba1c and glucose_fasting have high correlation
# so we drop the features with the lower correlation with the target labe
new = numerical_features_interestname.copy()
new.remove("hba1c")
new.remove("systolic_bp")
new.remove("diabetes_risk_score")
features_in_interest = features.copy()
features_in_interest.extend(new)
print(features_in_interest)

In [ ]:
X = df[features_in_interest]
X.head()

In [ ]:
y = df["diabetes_risk_score"]

# ***Data Preproccesing***

In [ ]:
from sklearn.model_selection import train_test_split

# Replace values that are smaller than or equal to 0, and values that are equal to infinity with Nan
X.loc[X["age"] == np.inf, "age"] = np.nan
X.loc[X["age"] <= 0, "age"] = np.nan
# This line was incorrectly setting the entire row to NaN; it should only affect the 'age' column.
X.loc[(X["age"] % 1 != 0) & (~X["age"].isnull()), "age"] = np.nan

# Split the data into training 80%, testing 10%, and validation 10%
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.1, random_state=4, shuffle=True)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size = 1/9, random_state = 4, shuffle=True)

# Print the number of rows for the training, validation, and testing data (To double check that the split worked as expected)
print(f"X_train: {X_train.shape[0]}, y_train: {y_train.shape[0]}")
print(f"X_test: {X_test.shape[0]}, y_test: {y_test.shape[0]}")
print(f"X_val: {X_val.shape[0]}, y_val: {y_val.shape[0]}")

X_train.describe()

In [ ]:
from sklearn.impute import SimpleImputer

# median Imputer
imp = SimpleImputer(missing_values=np.nan ,strategy="median")

features = X.columns
# fit the imputer on the training data, and apply the imputation on the validation data
X_train = imp.fit_transform(X_train)
X_test = imp.transform(X_test)
X_val = imp.transform(X_val)

X_train = pd.DataFrame(data=X_train,columns=features)
X_test = pd.DataFrame(data=X_test,columns=features)
X_val = pd.DataFrame(data=X_val, columns=features)

X_train.describe()

In [ ]:
# Apply cube root transformation to "physical acitivity minute per week" because the data is skewed
# cube root is used because log transformation made the distribution negatively skewed
# this is because cube root transformatin is weaker than log transformation
feature = "physical_activity_minutes_per_week"
sns.displot(X_train[feature],kde=True)
plt.title(f"Histogram of feature {feature} before transformation")
plt.xlabel(feature)
plt.ylabel("Frequency")
plt.show()

X_train["physical_activity_minutes_per_week"] = np.cbrt(X_train["physical_activity_minutes_per_week"])
X_test["physical_activity_minutes_per_week"] = np.cbrt(X_test["physical_activity_minutes_per_week"])
X_val["physical_activity_minutes_per_week"] = np.cbrt(X_val["physical_activity_minutes_per_week"])

feature = "physical_activity_minutes_per_week"
sns.displot(X_train[feature],kde=True)
plt.title(f"Histogram of feature {feature} after transformation")
plt.xlabel(feature)
plt.ylabel("Frequency")
plt.show()

In [ ]:
from sklearn.preprocessing import StandardScaler

# Use Standardization to normalize the data
scaler = StandardScaler()
features_to_scale = X_train.columns.tolist()
features_to_scale.remove("family_history_diabetes")

# seperate the family_history_diabetes label since it is a binary categorical feature and should not be scaled
diabetes_history = X_train['family_history_diabetes']
X_train_scaled = X_train[features_to_scale]
X_train_scaled = scaler.fit_transform(X_train_scaled)
X_train_scaled = pd.DataFrame(data=X_train_scaled, columns=features_to_scale)
X_train = pd.concat([diabetes_history, X_train_scaled],axis = 1)


# repeat for the validation and training datasets
diabetes_history = X_val['family_history_diabetes']
X_val_scaled = X_val[features_to_scale]
X_val_scaled = scaler.transform(X_val_scaled)
X_val_scaled = pd.DataFrame(data=X_val_scaled, columns=features_to_scale)
X_val = pd.concat([diabetes_history, X_val_scaled],axis = 1)

diabetes_history = X_test['family_history_diabetes']
X_test_scaled = X_test[features_to_scale]
X_test_scaled = scaler.transform(X_test_scaled)
X_test_scaled = pd.DataFrame(data=X_test_scaled, columns=features_to_scale)
X_test = pd.concat([diabetes_history, X_test_scaled],axis = 1)

In [ ]:
# plot a distribution graph of all the features after data preproccesing
for feature in features_in_interest:
  sns.displot(X_train[feature],kde=True)
  plt.show()

In [ ]:
# show that no features contain missing values
X_train.isnull().sum()

# ***Modelling and Model Selection***

In [ ]:
from tensorflow import keras
from keras import layers

# model 1 has two hidden layers, the first with 32 neurons and the second with 8 neurons
# relu is used in the hidden layers since its a common practice
# relu is used in the output layer since risk score is a non-negative value
model1 = keras.Sequential([
    layers.Dense(32, activation='relu', input_shape=(X_train.shape[1],)),
    layers.Dropout(0.1),
    layers.Dense(8, activation='relu'),
    layers.Dropout(0.1),
    layers.Dense(1, activation='relu')
])

# use adam optimizer, and use mse as the loss function
model1.compile(optimizer='adam',
              loss='mean_squared_error',
              metrics=['mean_absolute_error'])
model1.summary()

In [ ]:
# repeat the process for the second model
# the second model contains three hidden layers:
# the first hidden layer has 128 neurons, the second has 64, and the last has 32
model2 = keras.Sequential([
    layers.Dense(128, activation='relu', input_shape=(X_train.shape[1],)),
    layers.Dropout(0.1),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.1),
    layers.Dense(32, activation='relu'),
    layers.Dropout(0.1),
    layers.Dense(1, activation='relu')
])

model2.compile(optimizer='adam',
              loss='mean_squared_error',
              metrics=['mean_absolute_error'])
model2.summary()

In [ ]:
#model 3 has two hidden layers
# the first hidden layer contains 128 neurons, and the last hidden layers has 64 neurons
model3 = keras.Sequential([
    layers.Dense(128, activation='relu', input_shape=(X_train.shape[1],)),
    layers.Dropout(0.1),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.1),
    layers.Dense(1, activation='relu')
])
model3.compile(optimizer='adam',
              loss='mean_squared_error',
              metrics=['mean_absolute_error'])
model3.summary()

In [ ]:
from keras.callbacks import EarlyStopping
# use early stopping to stop the training process if the validation loss (validation mse) does not improve
# If, starting from epoch 20, the validation mse does not improve after 10 epochs, stop training
early = EarlyStopping(mode="min",monitor='val_loss', restore_best_weights=True, patience=10, start_from_epoch=20)
history1 = model1.fit(X_train, y_train, epochs=200, validation_data=(X_val, y_val),callbacks=[early])

In [ ]:
# repeat for model 2
early = EarlyStopping(mode="min",monitor='val_loss', restore_best_weights=True, patience=10, start_from_epoch=20)
history2 = model2.fit(X_train, y_train, epochs=200, validation_data=(X_val, y_val),callbacks=[early])

In [ ]:
# repeat for model 3
early = EarlyStopping(mode="min",monitor='val_loss', restore_best_weights=True, patience=10, start_from_epoch=20)
history3 = model3.fit(X_train, y_train, epochs=200, validation_data=(X_val, y_val),callbacks=[early])

In [ ]:
import matplotlib.pyplot as plt

# plot the training history for each model across epochs
plt.plot(history1.history['mean_absolute_error'], label='mean_absolute_error')
plt.plot(history1.history['val_mean_absolute_error'], label='val_mean_absolute_error')
plt.xlabel('Epochs')
plt.ylabel('Mean Absolute Error1')
plt.legend()
plt.show()

plt.plot(history2.history['mean_absolute_error'], label='mean_absolute_error')
plt.plot(history2.history['val_mean_absolute_error'], label='val_mean_absolute_error')
plt.xlabel('Epochs')
plt.ylabel('Mean Absolute Error2')
plt.legend()
plt.show()

plt.plot(history3.history['mean_absolute_error'], label='mean_absolute_error')
plt.plot(history3.history['val_mean_absolute_error'], label='val_mean_absolute_error')
plt.xlabel('Epochs')
plt.ylabel('Mean Absolute Error3')
plt.legend()
plt.show()

In [ ]:
# find the minimum validation mae for the three models
min1 = np.min(history1.history['val_mean_absolute_error'])
min2 = np.min(history2.history['val_mean_absolute_error'])
min3 = np.min(history3.history['val_mean_absolute_error'])

print(f"Minimum of validation error of model 1: {min1}")
print(f"Minimum of validation error of model 2: {min2}")
print(f"Minimum of validation error of model 3: {min3}")

# ***Hyperparameter Tuning***

In [ ]:
from keras.optimizers import Adam, RMSprop

# this function will return a model with the specified optimizer and learning_rate
def create_model(learning_rate=0.001, optimizer_type='adam'):
    #model 3 architecture, since its the best performing model
    test_model = keras.Sequential([
        layers.Input(shape=(X_train.shape[1],)),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.1),
        layers.Dense(64, activation='relu'),
        layers.Dropout(0.1),
        layers.Dense(1, activation='relu')
    ])

    if optimizer_type == 'adam':
        optimizer = Adam(learning_rate=learning_rate)
    else:
        optimizer = RMSprop(learning_rate=learning_rate)

    test_model.compile(optimizer=optimizer,
                  loss='mean_squared_error',
                  metrics=['mean_absolute_error'])
    return test_model

In [ ]:
# these are the hyperparameters that will be tuned
# learning rate, batch size and optimizer
params = {
    'learning_rate': [0.00005,0.0001, 0.0005],
    'batch_size': [16,32,64],
    'optimizer_type': ['adam', 'rmsprop']
}

# create variables that will hold the best_parameters and their score
# as well a dicationary to hold the score for each hyperparameter combination
best_params = dict()
best_score = 1e5

param_scores = dict()

# go through each combination of hyperparameter
for lr in params['learning_rate']:
    for batch_size in params['batch_size']:
        for optimizer in params['optimizer_type']:
            # set up the hyperparamters and their values as a dictionary
            param = {'learning_rate': lr, 'batch_size': batch_size, 'optimizer_type': optimizer}
            print(f"training model on hyperparameters {param}")
            # create a model on the learning rate and optimizer, and fit it with the specified batch size
            # also we will use early stopping for this stage
            model = create_model(learning_rate=lr, optimizer_type=optimizer)
            early = EarlyStopping(mode="min",monitor='val_loss', restore_best_weights=True, patience=5, start_from_epoch=10)
            history = model.fit(X_train, y_train, epochs=100, validation_data=(X_val, y_val),callbacks=[early],verbose=0,batch_size=batch_size)
            # find the minimum validation mae for the hyperparameter combination
            # and save the hyperparameter and its score to the dictionary
            min_val = np.min(history.history['val_mean_absolute_error'])
            print(f"the minimum validation error for {param} is:")
            print(min_val)
            param_scores[tuple(param.items())] = min_val
            # if we have found a better hyperparamter combination, save it to best_score
            if min_val < best_score:
                best_score = min_val
                best_params = param
print(best_params)
print(best_score)

In [ ]:
# get the best hyperparameters and fit the model one more time on the hyperparameters
lr = best_params['learning_rate']
optimizer = best_params['optimizer_type']
batch_size = best_params['batch_size']

final_model = create_model(optimizer_type=optimizer,learning_rate=lr)
early = EarlyStopping(mode="min",monitor='val_loss', restore_best_weights=True, patience=10, start_from_epoch=20)
history = final_model.fit(X_train,y_train,validation_data=(X_val,y_val),epochs=200,callbacks=[early],batch_size=batch_size,verbose=0)

In [ ]:
plt.plot(history.history['mean_absolute_error'], label='mean_absolute_error')
plt.plot(history.history['val_mean_absolute_error'], label='val_mean_absolute_error')
plt.xlabel('Epochs')
plt.ylabel('Mean Absolute Error')
plt.legend()
plt.show()

# ***Final test results and model saving***

In [ ]:
test_mse, test_mae = final_model.evaluate(X_test,y_test)

print(f"the mean absolute error of the best model on testing data is {test_mae}")

error = 100 * test_mae/(y.max() - y.min())

print(f"that is approximately {error}% error")

In [ ]:
final_model.save("saved_model.keras")

import pickle

with open("scaler.pkl", "wb") as file:
    pickle.dump(scaler,file)

with open("imputer.pkl", "wb") as file:
    pickle.dump(imp,file)